<a href="https://colab.research.google.com/github/ibrahimsaqib-ai/Introduction-to-AI-Labs/blob/main/Lab-01/scrapingtask1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Web Scraping Wikipedia: List of Countries by Population

This script will scrape data from the Wikipedia page "List of countries by population". We will use the `requests` library to fetch the webpage content, `BeautifulSoup` to parse the HTML and extract the table data, and `pandas` to store and display the data in a structured format.

In [ ]:
# Install necessary libraries
!pip install requests beautifulsoup4 pandas

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [ ]:
# Define the URL of the Wikipedia page
url = 'https://en.wikipedia.org/wiki/List_of_countries_by_population'

# Define a custom User-Agent header to mimic a web browser for pd.read_html
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

print("URL and custom headers configured for web scraping.")

URL and custom headers configured for web scraping.


In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

# Define the URL of the Wikipedia page
url = 'https://en.wikipedia.org/wiki/List_of_countries_by_population'

# Define a custom User-Agent header to mimic a web browser for pd.read_html
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

# Use pandas to read HTML tables directly from the URL
try:
    # First, fetch the page content using requests to ensure we can access it
    response = requests.get(url, headers=headers)
    response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
    html_content = response.text

    print("Successfully fetched webpage content with requests.")

    # Now, try to use pandas to read HTML tables from the fetched content
    # Based on previous inspection, the main table is the first one found.
    tables = pd.read_html(html_content, header=0, flavor='bs4')

    if tables:
        df = tables[0] # Select the first table, which was identified as the correct one
        print(f"Found {len(tables)} tables on the page. Using the first table as the main population data.")

        # Initial cleaning of column names: remove reference indicators like '[A]' and leading/trailing spaces
        df.columns = [col.split('[')[0].strip() for col in df.columns]

        # Clean 'Population' column: remove commas and extract numeric part
        if 'Population' in df.columns:
            # Use a more robust regex to extract numbers, handling cases like '1,000,000 (est.)'
            # Replace non-numeric, non-period characters, then extract the first numeric sequence
            df['Population'] = df['Population'].astype(str).str.replace(',', '', regex=False) \
                                             .str.extract(r'(\d+(?:\.\d+)?)').fillna('0').astype(float)
            # The regex `(\d+(?:\.\d+)?)` captures integers or decimal numbers.
            # `fillna('0')` handles cases where extract finds no digits.

        # Filter out rows that might not be actual countries (e.g., footnotes, sub-headers)
        # A simple check for 'Country / Dependency' column having sufficient length.
        # The column name might vary, let's look for 'Location' based on the identified table head.
        if 'Location' in df.columns:
            # Filter out rows where 'Location' might be empty or a summary row like 'World'
            # We want to keep actual countries/dependencies.
            df = df[df['Location'].str.len() > 1] # Ensure location name is not empty
            df = df[df['Location'].str.lower() != 'world'] # Remove the 'World' aggregate row if present
            df = df.reset_index(drop=True)

        print("Successfully created and cleaned DataFrame.")
        display(df.head())
        print(f"DataFrame contains {len(df)} rows after cleaning.")
    else:
        print("No tables were found in the fetched HTML content using pandas.read_html.")

except requests.exceptions.HTTPError as errh:
    print(f"HTTP Error: {errh}")
    print("This means the website might still be blocking access despite the User-Agent.")
except requests.exceptions.ConnectionError as errc:
    print(f"Error Connecting: {errc}")
except requests.exceptions.Timeout as errt:
    print(f"Timeout Error: {errt}")
except requests.exceptions.RequestException as err:
    print(f"Something unexpected happened with the request: {err}")
except Exception as e:
    print(f"An error occurred during table processing: {e}")
    print("This might be due to a change in the table's HTML structure or an issue with pandas parsing.")

Successfully fetched webpage content with requests.


/tmp/ipykernel_2927/1528558900.py:24: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(html_content, header=0, flavor='bs4')


Found 4 tables on the page. Using the first table as the main population data.
Successfully created and cleaned DataFrame.


,Location,Population,% of world,Date,Source (official or from the United Nations),Notes
0,India,1.429404e+09,17.3%,1 Jul 2026,Official projection[4],[b]
1,China,1.404890e+09,17.0%,31 Dec 2025,Official estimate[5],[c]
2,United States,3.417849e+08,4.1%,1 Jul 2025,Official estimate[6],[d]
3,Indonesia,2.883151e+08,3.5%,31 Dec 2025,National annual projection[7],NaN
4,Pakistan,2.414994e+08,2.9%,1 Mar 2023,2023 census result[8],[e]


DataFrame contains 239 rows after cleaning.


In [ ]:
# Save the DataFrame to an Excel file
output_filename = 'population_data.xlsx'
df.to_excel(output_filename, index=False)

print(f"Data successfully saved to {output_filename}")

Data successfully saved to population_data.xlsx
